In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
from datetime import datetime
import yfinance as yf
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()

HF_LOGIN_KEY = os.getenv("HF_LOGIN_KEY")
if HF_LOGIN_KEY:
    from huggingface_hub import login
    login(HF_LOGIN_KEY)
    print("Logged in to Hugging Face Hub successfully.")
else:
    print("HF_LOGIN_KEY not found in environment variables. Please set it to log in to Hugging Face Hub.")

c:\Users\Santi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Logged in to Hugging Face Hub successfully.


In [3]:
def multiply(a: float, b: float) -> str:
    result = a * b
    return f"The result of {a} × {b} is {result}."

def calculate_inflation_impact(amount: float, months: int, annual_inflation_rate: float) -> str:
    monthly_rate = (1 + annual_inflation_rate / 100) ** (1 / 12) - 1
    future_equivalent = amount * (1 + monthly_rate) ** months
    purchasing_power_loss = future_equivalent - amount
    effective_value = amount - purchasing_power_loss

    return (
        f"With an annual inflation rate of {annual_inflation_rate:.2f}%, "
        f"{amount:.2f} pesos today will have the purchasing power of "
        f"{effective_value:.2f} pesos after {months} month(s). "
        f"That is a loss of {purchasing_power_loss:.2f} pesos in real value."
    )

In [4]:
print(multiply(100, 0.85))

The result of 100 × 0.85 is 85.0.


In [5]:
print(calculate_inflation_impact(1000, 5, 4.5))

With an annual inflation rate of 4.50%, 1000.00 pesos today will have the purchasing power of 981.49 pesos after 5 month(s). That is a loss of 18.51 pesos in real value.


In [6]:
import os
from dotenv import load_dotenv
import requests
import datetime

load_dotenv()

BANXICO_TOKEN = os.getenv("BANXICO_TOKEN")

CETES_SERIES = {
    28:  "SF43936",
    91:  "SF43939",
    182: "SF43942",
    364: "SF43945",
    728: "SF349785",
}


#@tool("get_cetes_rate")
def get_cetes_rate(term_days: int, date: str | None = None) -> str:
    """
    Fetches the CETES rate for a given term in days from Banxico.
    Valid terms are: 28, 91, 182, 364, 728.
    """
    if term_days not in CETES_SERIES:
        valid = ", ".join(str(k) for k in CETES_SERIES)
        return f"Invalid term '{term_days}'. Valid options are: {valid}."

    series_id = CETES_SERIES[term_days]
    url = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{series_id}/datos"
    headers = {
        "Bmx-Token": BANXICO_TOKEN,
        "Content-Type": "application/json",
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        obs_list = response.json()["bmx"]["series"][0]["datos"]

        if date is None:
            obs = obs_list[-1]
        else:
            target = datetime.datetime.strptime(date, "%Y-%m-%d")
            obs = min(
                obs_list,
                key=lambda o: abs(datetime.datetime.strptime(o["fecha"], "%d/%m/%Y") - target),
            )

        fecha = datetime.datetime.strptime(obs["fecha"], "%d/%m/%Y").strftime("%Y-%m-%d")
        value = float(obs["dato"])
        label = f"nearest to {date}" if date else "most recent"
        return f"The CETES {term_days}-day rate ({label}) is {value:.4f}% as of {fecha}."

    except ValueError:
        return f"Invalid date format '{date}'. Please use YYYY-MM-DD (e.g. 2024-01-15)."
    except Exception as exc:
        return f"Error fetching CETES {term_days}-day rate: {exc}"

In [7]:
get_cetes_rate(28)

'The CETES 28-day rate (most recent) is 6.4900% as of 2026-05-07.'

In [8]:
TIIE_SERIES = {
    28:  "SF43783",
    91:  "SF43878",
    182: "SF111916",
}


def get_tiie_rate(term_days: int, date: str | None = None) -> str:

    if term_days not in TIIE_SERIES:
        valid = ", ".join(str(k) for k in TIIE_SERIES)
        return f"Invalid term '{term_days}'. Valid options are: {valid}."

    series_id = TIIE_SERIES[term_days]
    url = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{series_id}/datos"
    headers = {
        "Bmx-Token": BANXICO_TOKEN,
        "Content-Type": "application/json",
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        obs_list = response.json()["bmx"]["series"][0]["datos"]

        if date is None:
            obs = obs_list[-1]
        else:
            target = datetime.datetime.strptime(date, "%Y-%m-%d")
            obs = min(
                obs_list,
                key=lambda o: abs(datetime.datetime.strptime(o["fecha"], "%d/%m/%Y") - target),
            )

        fecha = datetime.datetime.strptime(obs["fecha"], "%d/%m/%Y").strftime("%Y-%m-%d")
        value = float(obs["dato"])
        label = f"nearest to {date}" if date else "most recent"
        return f"The TIIE {term_days}-day rate ({label}) is {value:.4f}% as of {fecha}."

    except ValueError:
        return f"Invalid date format '{date}'. Please use YYYY-MM-DD (e.g. 2024-01-15)."
    except Exception as exc:
        return f"Error fetching TIIE {term_days}-day rate: {exc}"

In [9]:
get_tiie_rate(28)

'The TIIE 28-day rate (most recent) is 6.7559% as of 2026-05-12.'

In [11]:
get_tiie_rate(91, "2024-06-01")

'The TIIE 91-day rate (nearest to 2024-06-01) is 11.3926% as of 2024-05-31.'